# Ch.6 — Cold Start & Production Serving

> **The story.** The cold start problem was formally described in **2002** by Schein et al., but the production solutions emerged from hard lessons at scale. **Netflix's 2012** engineering blog ("It's All A/B") changed how the industry thought about evaluation: every algorithm change must run through live A/B tests before deployment, because real users behave nothing like held-out test sets. In **2015**, Spotify's _Discover Weekly_ solved cold start for new songs by combining collaborative filtering with raw audio content analysis: if a song _sounds like_ what you like, recommend it even with zero plays. **Lin et al. (2019)** formalised the warm-up transition in "Warm Up Cold-Start Advertisements," showing that blending content features with early collaborative signals dramatically outperforms either alone — and that the blend weight should shift automatically as evidence accumulates. The production gap that opens here is between research and reality: an offline 87% HR@10 in a notebook is not the same as 87% HR@10 in production, where new users arrive every second with zero history, new items launch with zero ratings, and the SLA is **100ms** hard ceiling.
>
> **Where you are in the curriculum.** This is **Chapter 6 — the final chapter** of the Recommender Systems track. You have built FlixAI from a 42% popularity baseline through CF (68%), matrix factorization (78%), neural embeddings (83%), to a hybrid DCN system (**87% HR@10**). The accuracy constraint is satisfied — but 15% of monthly traffic is new signups with zero watch history, and every new movie launches with zero ratings on day one. This chapter solves cold start via content-based initialisation and bandit exploration, builds the production two-stage retrieval pipeline, designs A/B testing, and delivers monitoring. By the end, **all 5 FlixAI constraints are satisfied** and the system is live in production.
>
> **Notation.** $\mu_i$ — estimated CTR for arm $i$; $n_i$ — pulls for arm $i$; $N$ — total pulls; $\text{UCB1}(i) = \mu_i + \sqrt{2 \ln N / n_i}$ — upper confidence bound; $\mathbf{g} \in \mathbb{R}^{18}$ — user genre preference vector; $K$ — ANN candidate count; $\delta$ — minimum detectable effect; $\epsilon$ — exploration rate; $\gamma$ — epsilon decay (0.99).

---

## §0 · The Challenge — Where We Are

> **The mission**: Launch **FlixAI** — >85% HR@10 across all 5 constraints. This is the final blocker.

**What Ch.5 unlocked:** Hybrid DCN = **87% HR@10** [Done]. MMR diversity [Done]. Explainability [Done]. **Two blockers remain:**

1. **Cold start (15% of traffic):** Sarah signs up → zero watch history → hybrid model has no user embedding → system defaults to generic popularity list → Sarah churns in 3 sessions.

2. **Latency SLA — <100ms hard ceiling:** Ch.5's hybrid model scores all 1,682 items per request, taking ~350ms. At million-user scale, p99 latency exceeds 2 seconds — unacceptable.

```
Production pipeline target (< 100ms SLA):
───────────────────────────────────────────────────────────────────
Request  →  ANN retrieval  →  Hybrid ranker  →  MMR re-rank  →  Serve
            (FAISS/HNSW)      (DCN scoring)     (diversity)     (JSON)
            1,682 → 100        100 → 10           top-10         10 recs
            ~10ms               ~70ms              ~10ms          ~10ms
                                                         total: ~100ms
───────────────────────────────────────────────────────────────────
```

**Dataset:** MovieLens 100k | **Task:** Cold start strategies + production pipeline | **Outcome:** All 5 constraints satisfied


## §1 · The Core Idea

Cold start is the **chicken-and-egg problem of recommendation**: we need ratings data to personalise, but we need to recommend to collect ratings data. The hybrid model from Ch.5 is helpless: no interaction history → no user embedding → no personalisation → user churns before we learn anything.

Three strategies break the cycle, and they compose into a single decision pipeline:

```mermaid
flowchart TD
    REQ["New request"] --> HASEMB{"User has\nembedding?"}
    HASEMB -- "Yes (warm user)" --> HYBRID["Full Hybrid DCN\n87% HR@10"]
    HASEMB -- "No (cold user)" --> HASSURVEY{"Onboarding\nsurvey done?"}
    HASSURVEY -- "Yes" --> CONTENTINIT["Content-based init\n(genre prefs → embedding)"]
    HASSURVEY -- "No" --> POPULAR["Popularity fallback\n(global top-10)"]
    CONTENTINIT --> BANDIT["UCB Bandit\n(explore + exploit)"]
    POPULAR --> BANDIT
    BANDIT --> THRESHOLD{"Interactions\naccumulated?"}
    THRESHOLD -- "< 10" --> BANDITSERVE["Bandit-served\n(exploration heavy)"]
    THRESHOLD -- "10–50" --> WARMBLEND["Warm blend\nBandit + Hybrid"]
    THRESHOLD -- "> 50" --> HYBRID

    style REQ fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style HASEMB fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style HYBRID fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style POPULAR fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style CONTENTINIT fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style BANDIT fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style BANDITSERVE fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style WARMBLEND fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Strategy 1 — Content-based initialisation:** Even five onboarding answers give enough signal to serve a reasonable top-10 from day one. For new items, use genre, director, and cast metadata until collaborative signals accumulate.

**Strategy 2 — Bandit exploration:** Treat each interaction as a pull of a multi-armed bandit. UCB1 scores each item as _estimated reward + uncertainty bonus_. Items shown rarely have a large bonus — the system explores them until their true quality is measured.

**Strategy 3 — Two-stage retrieval:** ANN index (10ms, 100 candidates) + DCN ranker (70ms, top-10) + MMR (10ms) = **<100ms total**.

> **Optional depth:** $\text{UCB1}(i) = \mu_i + \sqrt{2 \ln N / n_i}$ — reward estimate $\mu_i$ plus exploration bonus that shrinks as arm $i$ is pulled more often. Items never shown have $n_i = 0$ → infinite bonus → shown at least once.


In [ ]:
# TODO: Implement this cell
#  (Imports)
#
# Steps:
# 1. Imports
# 2. Compute `SEED` using `set_theme()`
# 3. Process data
#
# Hint:
#    # implement using the APIs described above

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
np.random.seed(SEED)

print("Libraries loaded.")

In [ ]:
# TODO: Implement this cell
#  (Load MovieLens 100k)
#
# Steps:
# 1. Load MovieLens 100k
# 2. Compute `ratings` using `read_csv()`
# 3. Compute `movies` using `read_csv()`
# 4. Compute `genre_cols`
# 5. Compute `n_users`
# 6. Process data
#
# Hint:
#    ratings = pd.read_csv(???)
#    movies = pd.read_csv(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Load MovieLens 100k ───────────────────────────────────────────────────
url = "https://files.grouplens.org/datasets/movielens/ml-100k/"

ratings = pd.read_csv(
    url + "u.data", sep="\t", names=["user_id", "item_id", "rating", "timestamp"]
)

movies = pd.read_csv(
    url + "u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    names=[
        "item_id",
        "title",
        "release_date",
        "video_release",
        "url",
        "unknown",
        "Action",
        "Adventure",
        "Animation",
        "Children",
        "Comedy",
        "Crime",
        "Documentary",
        "Drama",
        "Fantasy",
        "Film-Noir",
        "Horror",
        "Musical",
        "Mystery",
        "Romance",
        "Sci-Fi",
        "Thriller",
        "War",
        "Western",
    ],
    usecols=range(24),
)

genre_cols = [
    "Action",
    "Adventure",
    "Animation",
    "Children",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Fantasy",
    "Film-Noir",
    "Horror",
    "Musical",
    "Mystery",
    "Romance",
    "Sci-Fi",
    "Thriller",
    "War",
    "Western",
]

n_users = ratings["user_id"].max()
n_items = ratings["item_id"].max()

print(f"Users: {n_users}  Items: {n_items}  Ratings: {len(ratings):,}")

In [ ]:
# TODO: Implement this cell
#  (Simulate Cold Start Scenario)
#
# Steps:
# 1. Simulate Cold Start Scenario
# 2. Compute `cold_ratings` using `isin()`
# 3. Aggregate data into `cold_sorted` -- use `groupby()`
# 4. Process data
#
# Hint:
#    cold_users = np.random.choice(???)
#    cold_sorted = cold_ratings.sort_values(???)
#    onboarding = cold_sorted.groupby(???)
#    cold_test = cold_sorted.groupby(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Simulate Cold Start Scenario ──────────────────────────────────────────
# Pick 100 users as "new" users: hold out ALL their ratings
np.random.seed(SEED)
cold_users = np.random.choice(range(1, n_users + 1), size=100, replace=False)

cold_ratings = ratings[ratings["user_id"].isin(cold_users)].copy()
warm_ratings = ratings[~ratings["user_id"].isin(cold_users)].copy()

# Simulate onboarding: cold users' first 3 ratings are "onboarding"
cold_sorted = cold_ratings.sort_values("timestamp")
onboarding = cold_sorted.groupby("user_id").head(3)
cold_test = cold_sorted.groupby("user_id").tail(1)

print(f"Cold start users: {len(cold_users)}")
print(f"Onboarding ratings: {len(onboarding)} (3 per user)")
print(f"Cold test ratings: {len(cold_test)} (1 per user)")
print(f"Warm training ratings: {len(warm_ratings):,}")

In [ ]:
def content_based_recommend(user_genre_prefs, item_genres_df, genre_cols, n_recs=10):
    """
    TODO #4: Implement `content_based_recommend()`.

    Steps:
    1. Content-Based Fallback
    2. Process data
    3. Call `norm()` to produce the result
    4. Call `argsort()` to produce the result
    5. Define helper function `get_genre_prefs_from_ratings()`
    6. Compute `sample_user`

    Hint:
    user_vec = np.array(???)
    norms = np.linalg.norm(???)
    top_items = np.argsort(???)
    merged = user_ratings.merge(???)

    Returns: list(range(1, n_recs + 1))
    """
    raise NotImplementedError("TODO: implement content_based_recommend()")


def get_genre_prefs_from_ratings(user_ratings, movies_df, genre_cols):
    """
    TODO #4: Implement `get_genre_prefs_from_ratings()`.

    Steps:
    1. Content-Based Fallback
    2. Process data
    3. Call `norm()` to produce the result
    4. Call `argsort()` to produce the result
    5. Define helper function `get_genre_prefs_from_ratings()`
    6. Compute `sample_user`

    Hint:
    user_vec = np.array(???)
    norms = np.linalg.norm(???)
    top_items = np.argsort(???)
    merged = user_ratings.merge(???)

    Returns: list(range(1, n_recs + 1))
    """
    raise NotImplementedError("TODO: implement get_genre_prefs_from_ratings()")

## §2 · Bandit Algorithms for Exploration

A new user with 3 onboarding ratings knows they like Sci-Fi — but _which_ sci-fi? Pure exploitation always shows the highest-scoring items. The problem: if you only show items you're confident about, you never learn about the others. Bandits solve this by giving a bonus to items you know _little_ about, ensuring the system explores until uncertainty is resolved.

Think of it as a librarian trying to please a new patron: instead of always recommending the bestseller, occasionally suggest a lesser-known title and note whether they enjoy it. As evidence accumulates, the librarian relies more on what they've learned.

### Epsilon-Greedy — simple but effective

With probability $\epsilon$, explore (random item); with probability $1-\epsilon$, exploit (model's best). $\epsilon$ decays over time: $\epsilon_t = \max(\epsilon_{\min}, \epsilon_0 \cdot \gamma^t)$.

> **Optional depth:** $\epsilon = 0.3$ on day one (30% random) decays to $\epsilon_{\min} = 0.05$ after ~50 interactions. Simple, cheap, works well when the reward landscape is stationary.

### Upper Confidence Bound (UCB) — principled exploration

$$\text{UCB1}(i) = \mu_i + c\sqrt{\frac{\ln t}{n_i}}$$

> **Optional depth:** $\mu_i$ — estimated reward (historical CTR for item $i$); $n_i$ — number of times item $i$ has been shown; $t$ — total impressions; $c$ — exploration coefficient. The bonus $c\sqrt{\ln t / n_i}$ shrinks as $n_i$ grows. Items never shown have $n_i = 0$ → infinite bonus → guaranteed to be shown at least once. UCB minimises cumulative regret logarithmically — mathematically optimal for stationary rewards.


In [ ]:
# TODO: Implement this cell
#  (Epsilon-Greedy Bandit)
#
# Steps:
# 1. Epsilon-Greedy Bandit
# 2. Define helper function
# 3. Call `random()` to produce the result
# 4. Call `tolist()` to produce the result
# 5. Define helper function
# 6. Process data
#
# Hint:
#    item_rewards = np.zeros(???)
#    item_counts = np.zeros(???)
#    model_scores = model_scores.copy(???)
#    top = np.argsort(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Epsilon-Greedy Bandit ─────────────────────────────────────────────────
class EpsilonGreedyBandit:
    def __init__(self, n_items, epsilon=0.3, decay=0.95, min_epsilon=0.05):
        self.n_items = n_items
        self.epsilon = epsilon
        self.decay = decay
        self.min_epsilon = min_epsilon
        self.item_rewards = np.zeros(n_items + 1)
        self.item_counts = np.zeros(n_items + 1)

    def select(self, model_scores, already_rated, n_recs=10):
        """Select items balancing exploitation and exploration."""
        model_scores = model_scores.copy()
        for r in already_rated:
            if r < len(model_scores):
                model_scores[r] = -np.inf
        model_scores[0] = -np.inf

        if np.random.random() < self.epsilon:
            # Explore: top-7 from model + 3 random
            n_exploit = max(1, n_recs - 3)
            n_explore = n_recs - n_exploit
            top = np.argsort(model_scores)[-n_exploit:][::-1]

            valid = np.where(model_scores > -np.inf)[0]
            explore = np.random.choice(
                valid, size=min(n_explore, len(valid)), replace=False
            )
            return np.concatenate([top, explore]).astype(int).tolist()[:n_recs]
        else:
            return np.argsort(model_scores)[-n_recs:][::-1].tolist()

    def update(self, item_id, reward):
        self.item_counts[item_id] += 1
        n = self.item_counts[item_id]
        self.item_rewards[item_id] += (reward - self.item_rewards[item_id]) / n

    def decay_epsilon(self):
        self.epsilon = max(self.min_epsilon, self.epsilon * self.decay)


print("EpsilonGreedyBandit defined.")

In [ ]:
# TODO: Implement this cell
#  (UCB Bandit)
#
# Steps:
# 1. UCB Bandit
# 2. Define helper function
# 3. Call `copy()` to produce the result
# 4. Process data
# 5. Call `argsort()` to produce the result
# 6. Define helper function
# 7. Process data
#
# Hint:
#    item_rewards = np.zeros(???)
#    item_counts = np.zeros(???)
#    ucb_scores = model_scores.copy(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── UCB Bandit ────────────────────────────────────────────────────────────
class UCBBandit:
    def __init__(self, n_items, c=1.5):
        self.n_items = n_items
        self.c = c
        self.item_rewards = np.zeros(n_items + 1)
        self.item_counts = np.zeros(n_items + 1)
        self.total_pulls = 0

    def select(self, model_scores, already_rated, n_recs=10):
        """Select items using UCB: predicted score + exploration bonus."""
        self.total_pulls += 1

        ucb_scores = model_scores.copy()
        for i in range(1, self.n_items + 1):
            if self.item_counts[i] > 0:
                exploration = self.c * np.sqrt(
                    np.log(self.total_pulls) / self.item_counts[i]
                )
            else:
                exploration = self.c * 10  # high bonus for unexplored
            ucb_scores[i] = model_scores[i] + exploration

        for r in already_rated:
            if r < len(ucb_scores):
                ucb_scores[r] = -np.inf
        ucb_scores[0] = -np.inf

        return np.argsort(ucb_scores)[-n_recs:][::-1].tolist()

    def update(self, item_id, reward):
        self.item_counts[item_id] += 1
        n = self.item_counts[item_id]
        self.item_rewards[item_id] += (reward - self.item_rewards[item_id]) / n


print("UCBBandit defined.")

In [ ]:
def evaluate_cold_strategy(strategy_name, rec_func, cold_test_df, k=10):
    """
    TODO #7: Implement `evaluate_cold_strategy()`.

    Steps:
    1. Set up: Simulate Cold Start with Different Strategies
    2. Aggregate data into `item_stats` -- use `groupby()`
    3. Compute `pop_scores` using `zeros()`
    4. Define helper function `evaluate_cold_strategy()`
    5. Define helper function `pop_strategy()`
    6. Define helper function `content_strategy()`
    7. Define helper function `eg_strategy()`
    8. Define helper function `ucb_strategy()`
    9. Compute `hr_pop`
    10. Call `Strategies()` to produce the result

    Hint:
    bandit_eg = EpsilonGreedyBandit(epsilon=???)
    bandit_ucb = UCBBandit(c=???)
    item_stats = warm_ratings.groupby(???)
    pop_scores = np.zeros(???)

    Returns: hits / len(cold_test_df)
    """
    raise NotImplementedError("TODO: implement evaluate_cold_strategy()")


def pop_strategy(uid):
    """
    TODO #7: Implement `pop_strategy()`.

    Steps:
    1. Set up: Simulate Cold Start with Different Strategies
    2. Aggregate data into `item_stats` -- use `groupby()`
    3. Compute `pop_scores` using `zeros()`
    4. Define helper function `evaluate_cold_strategy()`
    5. Define helper function `pop_strategy()`
    6. Define helper function `content_strategy()`
    7. Define helper function `eg_strategy()`
    8. Define helper function `ucb_strategy()`
    9. Compute `hr_pop`
    10. Call `Strategies()` to produce the result

    Hint:
    bandit_eg = EpsilonGreedyBandit(epsilon=???)
    bandit_ucb = UCBBandit(c=???)
    item_stats = warm_ratings.groupby(???)
    pop_scores = np.zeros(???)

    Returns: hits / len(cold_test_df)
    """
    raise NotImplementedError("TODO: implement pop_strategy()")


def content_strategy(uid):
    """
    TODO #7: Implement `content_strategy()`.

    Steps:
    1. Set up: Simulate Cold Start with Different Strategies
    2. Aggregate data into `item_stats` -- use `groupby()`
    3. Compute `pop_scores` using `zeros()`
    4. Define helper function `evaluate_cold_strategy()`
    5. Define helper function `pop_strategy()`
    6. Define helper function `content_strategy()`
    7. Define helper function `eg_strategy()`
    8. Define helper function `ucb_strategy()`
    9. Compute `hr_pop`
    10. Call `Strategies()` to produce the result

    Hint:
    bandit_eg = EpsilonGreedyBandit(epsilon=???)
    bandit_ucb = UCBBandit(c=???)
    item_stats = warm_ratings.groupby(???)
    pop_scores = np.zeros(???)

    Returns: hits / len(cold_test_df)
    """
    raise NotImplementedError("TODO: implement content_strategy()")


def eg_strategy(uid):
    """
    TODO #7: Implement `eg_strategy()`.

    Steps:
    1. Set up: Simulate Cold Start with Different Strategies
    2. Aggregate data into `item_stats` -- use `groupby()`
    3. Compute `pop_scores` using `zeros()`
    4. Define helper function `evaluate_cold_strategy()`
    5. Define helper function `pop_strategy()`
    6. Define helper function `content_strategy()`
    7. Define helper function `eg_strategy()`
    8. Define helper function `ucb_strategy()`
    9. Compute `hr_pop`
    10. Call `Strategies()` to produce the result

    Hint:
    bandit_eg = EpsilonGreedyBandit(epsilon=???)
    bandit_ucb = UCBBandit(c=???)
    item_stats = warm_ratings.groupby(???)
    pop_scores = np.zeros(???)

    Returns: hits / len(cold_test_df)
    """
    raise NotImplementedError("TODO: implement eg_strategy()")


def ucb_strategy(uid):
    """
    TODO #7: Implement `ucb_strategy()`.

    Steps:
    1. Set up: Simulate Cold Start with Different Strategies
    2. Aggregate data into `item_stats` -- use `groupby()`
    3. Compute `pop_scores` using `zeros()`
    4. Define helper function `evaluate_cold_strategy()`
    5. Define helper function `pop_strategy()`
    6. Define helper function `content_strategy()`
    7. Define helper function `eg_strategy()`
    8. Define helper function `ucb_strategy()`
    9. Compute `hr_pop`
    10. Call `Strategies()` to produce the result

    Hint:
    bandit_eg = EpsilonGreedyBandit(epsilon=???)
    bandit_ucb = UCBBandit(c=???)
    item_stats = warm_ratings.groupby(???)
    pop_scores = np.zeros(???)

    Returns: hits / len(cold_test_df)
    """
    raise NotImplementedError("TODO: implement ucb_strategy()")

In [ ]:
# TODO: Implement this cell
#  (Visualise Cold Start Strategies)
#
# Steps:
# 1. Visualise Cold Start Strategies
# 2. Plot results -- call `subplots()`
#
# Hint:
#    ax = plt.subplots(???)
#    bars = ax.bar(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Visualise Cold Start Strategies ───────────────────────────────────────
strategies = ["Popularity", "Content-Based", "ε-Greedy", "UCB"]
hr_values = [hr_pop, hr_content, hr_eg, hr_ucb]
colors = ["#95a5a6", "#3498db", "#27ae60", "#e74c3c"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(strategies, [h * 100 for h in hr_values], color=colors, edgecolor="white")
ax.set(
    ylabel="Hit Rate@10 (%)",
    title="Cold Start Strategies — New Users (3 onboarding ratings)",
)
for bar, val in zip(bars, hr_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{val*100:.1f}%",
        ha="center",
        fontsize=11,
        fontweight="bold",
    )
plt.tight_layout()
plt.savefig("img/cold_start_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

### What §1–§2 established — and what it still doesn't solve

[Done] **Cold start broken.** Content-based initialisation + UCB bandit provides meaningful personalisation from the very first session.

[Done] **Exploration–exploitation balanced.** UCB ensures the system learns about less-known items rather than permanently ignoring them.

[Done] **Warm-up transition defined.** The decision pipeline (cold → bandit → warm blend → full hybrid) handles every stage of a user's lifecycle.

**Still open:**

- **A/B validation:** Offline evaluation on held-out MovieLens data. Online A/B testing is required to confirm the lift in real user behavior before rollout.
- **Latency SLA at scale:** Two-stage ANN retrieval (§3 exercises) cuts this to <100ms.
- **Monitoring:** Without a monitoring dashboard, regressions in production are invisible until user churn spikes.


In [ ]:
# TODO: Implement this cell
#  (Epsilon Decay Simulation)
#
# Steps:
# 1. Epsilon Decay Simulation
# 2. Call `append()` to produce the result
# 3. Plot results -- call `subplots()`
# 4. Process data
#
# Hint:
#    ax = plt.subplots(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Epsilon Decay Simulation ──────────────────────────────────────────────
interactions = range(1, 101)
epsilon_history = []
eps = 0.3
decay = 0.95
min_eps = 0.05

for t in interactions:
    epsilon_history.append(eps)
    eps = max(min_eps, eps * decay)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(interactions, epsilon_history, color="#27ae60", linewidth=2)
ax.axhline(0.05, color="gray", linestyle="--", alpha=0.5, label="Min ε = 0.05")
ax.fill_between(interactions, epsilon_history, alpha=0.1, color="#27ae60")
ax.set(
    xlabel="Number of Interactions",
    ylabel="Exploration Rate (ε)",
    title="Epsilon Decay: Exploration → Exploitation",
)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("img/epsilon_decay.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"ε after 10 interactions: {epsilon_history[9]:.3f}")
print(f"ε after 50 interactions: {epsilon_history[49]:.3f}")
print(f"ε after 100 interactions: {epsilon_history[99]:.3f}")

In [ ]:
def ab_test_sample_size(p_control, min_lift, alpha=0.05, power=0.8):
    """
    TODO #10: Implement `ab_test_sample_size()`.

    Steps:
    1. A/B Test Power Analysis
    2. Define helper function `ab_test_significance()`
    3. Compute `n_per_group`

    Hint:
    z_alpha = stats.norm.ppf(???)
    z_beta = stats.norm.ppf(???)
    se = np.sqrt(???)

    Returns: int(np.ceil(n))
    """
    raise NotImplementedError("TODO: implement ab_test_sample_size()")


def ab_test_significance(hr_a, n_a, hr_b, n_b, alpha=0.05):
    """
    TODO #10: Implement `ab_test_significance()`.

    Steps:
    1. A/B Test Power Analysis
    2. Define helper function `ab_test_significance()`
    3. Compute `n_per_group`

    Hint:
    z_alpha = stats.norm.ppf(???)
    z_beta = stats.norm.ppf(???)
    se = np.sqrt(???)

    Returns: int(np.ceil(n))
    """
    raise NotImplementedError("TODO: implement ab_test_significance()")

In [ ]:
# TODO: Implement this cell
#  (Simulate A/B Test)
#
# Steps:
# 1. Simulate A/B Test
# 2. Compute `n_sim` using `A()`
# 3. Compute `results_a` using `binomial()`
# 4. Compute `p_values`
# 5. Call `mean()` to produce the result
# 6. Plot results -- call `subplots()`
# 7. Call `group()` to produce the result
#
# Hint:
#    results_a = np.random.binomial(???)
#    results_b = np.random.binomial(???)
#    ax = plt.subplots(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Simulate A/B Test ─────────────────────────────────────────────────────
np.random.seed(SEED)

# Simulate: Model A (HR=85%), Model B (HR=87%)
n_sim = 5000
hr_a_true = 0.85
hr_b_true = 0.87

results_a = np.random.binomial(1, hr_a_true, n_sim)
results_b = np.random.binomial(1, hr_b_true, n_sim)

# Running significance test
p_values = []
sample_sizes = range(100, n_sim + 1, 100)

for n in sample_sizes:
    result = ab_test_significance(results_a[:n].mean(), n, results_b[:n].mean(), n)
    p_values.append(result["p_value"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sample_sizes, p_values, color="#2980b9", linewidth=1.5)
ax.axhline(0.05, color="red", linestyle="--", alpha=0.7, label="p = 0.05 threshold")
ax.set(
    xlabel="Sample Size per Group",
    ylabel="p-value",
    title="A/B Test: When Does the 2% Lift Become Significant?",
)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale("log")
plt.tight_layout()
plt.savefig("img/ab_test_pvalue.png", dpi=150, bbox_inches="tight")
plt.show()

# Find when significance is first achieved
for n, p in zip(sample_sizes, p_values):
    if p < 0.05:
        print(f"First significant at n={n} per group (p={p:.4f})")
        break

## Final Progress Check — All Constraints

**Checkpoint:** FlixAI — hit@10 maintained at ≥87% while all 5 production constraints are now satisfied. The FlixAI system is production-ready.

| #   | Constraint     | Target                | Final Status                         |
| --- | -------------- | --------------------- | ------------------------------------ |
| 1   | ACCURACY       | >85% HR@10            | **87%** (Hybrid DCN, Ch.5)           |
| 2   | COLD START     | New users/items       | **[Done]** Content init + UCB bandit |
| 3   | SCALABILITY    | <100ms                | **[Done]** Two-stage ANN + ranker    |
| 4   | DIVERSITY      | Not just popular      | **[Done]** MMR re-ranking            |
| 5   | EXPLAINABILITY | "Because you liked X" | **[Done]** Content features          |

**GRAND CHALLENGE COMPLETE:** FlixAI is production-ready!

**Journey:** Popularity (42%) → CF (68%) → MF (78%) → NCF (82%) → Hybrid (87%) → Production [Done]


## Exercises

**Exercise 1 — LinUCB (Contextual Bandit)**
Implement LinUCB: use user features (age, gender, genre preferences) as context to personalise exploration. Compare HR@10 against basic UCB for cold-start users.

**Exercise 2 — Warm-Up Simulation**
Simulate a user's journey from cold start (0 ratings) to established (50 ratings). At each step, record which strategy is used and the cumulative hit rate. Plot the transition.

**Exercise 3 — Monitoring Dashboard**
Build a monitoring dashboard that tracks: daily HR@10, cold-start user fraction, exploration rate, and recommendation diversity. Use matplotlib to create a 2×2 subplot dashboard.


In [ ]:
# TODO: Implement this cell
#  (Exercise 1 scaffold — LinUCB)
#
# Steps:
# 1. Set up: Exercise 1 scaffold — LinUCB
# 2. Process data
#
# Hint:
#    # implement using the APIs described above

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Exercise 1 scaffold — LinUCB ─────────────────────────────────────────
# TODO: Implement contextual bandit using user features
# LinUCB: UCB(a|x) = θ_a^T x + c * sqrt(x^T A_a^{-1} x)

# class LinUCBBandit:
#     def __init__(self, n_items, n_features, c=1.5):
#         self.A = [np.eye(n_features) for _ in range(n_items + 1)]
#         self.b = [np.zeros(n_features) for _ in range(n_items + 1)]
#         ...
#     def select(self, user_context, ...):
#         ...

pass

In [ ]:
# TODO: Implement this cell
#  (Exercise 2 scaffold — Warm-Up Simulation)
#
# Steps:
# 1. Set up: Exercise 2 scaffold — Warm-Up Simulation
# 2. Process data
#
# Hint:
#    # implement using the APIs described above

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Exercise 2 scaffold — Warm-Up Simulation ─────────────────────────────
# TODO: Simulate a user going from 0 to 50 interactions
# Track: which strategy used (cold/warm/full), cumulative HR

# for n_interactions in range(0, 51):
#     if n_interactions < 10:
#         strategy = "cold_start"
#     elif n_interactions < 50:
#         strategy = "warm_blend"
#     else:
#         strategy = "full_hybrid"
#     # ... make recommendation, check if hit, record

pass

In [ ]:
# TODO: Implement this cell
#  (Exercise 3 scaffold — Monitoring Dashboard)
#
# Steps:
# 1. Set up: Exercise 3 scaffold — Monitoring Dashboard
# 2. Process data
#
# Hint:
#    axes = plt.subplots(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Exercise 3 scaffold — Monitoring Dashboard ───────────────────────────
# TODO: Create a 2x2 subplot dashboard showing:
# Top-left: Daily HR@10 over 30 days
# Top-right: Cold-start user fraction
# Bottom-left: Exploration rate (epsilon) over time
# Bottom-right: Recommendation diversity (ILD)

# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# ... simulate 30 days of metrics
# plt.savefig("img/monitoring_dashboard.png", dpi=150, bbox_inches="tight")

pass